# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. You can adapt it for any Croissant-conformant dataset.

### Dataset Source
The dataset is defined by this Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed. It is a lightweight dependency.
!pip install mlcroissant --quiet

## 1. Data Loading

Load the Croissant dataset and its metadata using `mlcroissant`. This will automatically fetch the schema and enable programmatic access to data, record sets, and field definitions.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")

## 2. Data Overview

Review available record sets, fields, their `@id`s, and key structural metadata. All objects are referenced by their `@id` for reliability and explicit referencing following the Croissant schema.

In [ ]:
# List all record sets in the dataset
print("Available Record Sets (by @id and label):\n")

record_sets = list(dataset.metadata.recordSets)
for record_set in record_sets:
    print(f"- @id: {record_set.id}\n  label: {getattr(record_set, 'name', '')}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for f in record_set.fields:
            print(f"    - @id: {f.id}, name: {getattr(f, 'name', '')}")
    print()
# Select one example record set for exploration
if record_sets:
    selected_record_set = record_sets[0]
    selected_record_set_id = selected_record_set.id
    print(f"Selected record set for further exploration: @{selected_record_set_id}")
else:
    raise RuntimeError("No record sets found in metadata.")

## 3. Data Extraction

Load data from record sets into Pandas DataFrames. This enables interactive exploration. 
We reference all sets and fields by their `@id` as required by the Croissant model.

In [ ]:
dataframes = {}

record_set_ids = [rs.id for rs in record_sets]

# Load available records for each record set into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print(f"Loaded DataFrames for record sets: {list(dataframes.keys())}")
example_df = dataframes[selected_record_set_id]
print(f"\nColumns in {selected_record_set_id}:\n", example_df.columns.tolist())
example_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's process the main table: filter records based on a numeric field, normalize, and aggregate. All fields will be referenced by their `@id`.

In [ ]:
# Pick a representative numeric field and a group/categorical field by @id from the overview above.
# Please update these with the actual @id of your numeric and group field from printouts above.
example_numeric_field = example_df.columns[0]  # Replace with actual field id if needed
for col in example_df.columns:
    # Try to auto-select a likely numeric field
    if example_df[col].dtype != object:
        example_numeric_field = col
        break

print(f"Using numeric field @id for filtering/normalization: {example_numeric_field}")

threshold = 10
filtered_df = example_df[example_df[example_numeric_field] > threshold]

print(f"Filtered records where {example_numeric_field} > {threshold} (first 5 shown):")
print(filtered_df.head())

# Normalize the chosen numeric field (Z-score)
_norm_col = f"{example_numeric_field}_normalized"
filtered_df[_norm_col] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
print(f"\nNormalized {example_numeric_field} for filtered records:")
print(filtered_df[[example_numeric_field, _norm_col]].head())

# Select a group field (e.g. protein family, or experimental condition) by @id; fall back to any string/categorical
group_field = None
for col in example_df.columns:
    if example_df[col].dtype == object and col != example_numeric_field:
        group_field = col
        break
if group_field is None and len(example_df.columns) > 1:
    group_field = example_df.columns[1]  # Fallback

print(f"\nGrouping field (by @id): {group_field}")

if group_field:
    grouped_df = filtered_df.groupby(group_field)[example_numeric_field].mean().reset_index()
    print(f"\nGrouped means of {example_numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the numeric field for filtered data and relationship with the identified group field. You can adapt or add additional plots as needed.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the normalized numeric field
plt.figure(figsize=(6, 4))
filtered_df[_norm_col].hist(bins=30, grid=False)
plt.title(f"Distribution of Normalized {example_numeric_field} (filtered)")
plt.xlabel(f"{example_numeric_field} (normalized)")
plt.ylabel("Count")
plt.show()

# Bar plot of group means if applicable
if group_field and 'grouped_df' in locals():
    top_n = min(10, len(grouped_df))
    plt.figure(figsize=(8, 4))
    plt.bar(grouped_df[group_field][:top_n], grouped_df[example_numeric_field][:top_n])
    plt.title(f"Mean {example_numeric_field} by {group_field} (top {top_n})")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {example_numeric_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- The FAIR^2 dataset mass spectrometry tables can be programmatically loaded and explored with Croissant via `mlcroissant`.
- Fields, columns, and record sets are referenced by their `@id`, which ensures reproducibility and enables robust pipeline integration.
- Basic EDA shows how to filter, normalize, and group quantitative values (such as protein abundances or peptide counts) and visualize the results.

You can extend this notebook with additional analyses, such as statistical tests, machine learning, or domain-specific modelling.

_For full dataset and metadata details, see [dataset page](https://sen.science/doi/10.71728/senscience.vq4a-28xa)._